In [5]:
# Cargar dataset del MEC
df_mec = pd.read_csv('../data/raw/mec_matriculas.csv')

print("✅ Dataset MEC cargado")
print("Shape:", df_mec.shape)
print("\nColumnas:", df_mec.columns.tolist())
df_mec.head()

✅ Dataset MEC cargado
Shape: (844, 11)

Columnas: ['anio', 'codigo_departamento', 'nombre_departamento', 'codigo_distrito', 'nombre_distrito', 'codigo_zona', 'nombre_zona', 'sector_o_tipo_gestion', 'anho_cod_geo', 'cantidad_matriculados_hombre', 'cantidad_matriculados_mujer']


,anio,codigo_departamento,nombre_departamento,codigo_distrito,nombre_distrito,codigo_zona,nombre_zona,sector_o_tipo_gestion,anho_cod_geo,cantidad_matriculados_hombre,cantidad_matriculados_mujer
0,2023,17,Alto Paraguay,4,Bahía Negra,1,Urbana,Oficial,NaN,244,205
1,2023,17,Alto Paraguay,4,Bahía Negra,2,Rural,Privado,NaN,11,18
2,2023,17,Alto Paraguay,4,Bahía Negra,2,Rural,Oficial,NaN,88,105
3,2023,17,Alto Paraguay,5,Carmelo Peralta,2,Rural,Oficial,NaN,347,337
4,2023,17,Alto Paraguay,5,Carmelo Peralta,1,Urbana,Oficial,NaN,271,223


## Ética, licencias y calidad de datos

### Fuentes y licencias
| Dataset | Fuente | Licencia | Fecha descarga |
|---|---|---|---|
| Matrícula MEC 2023 | datos.mec.gov.py | Licencia Pública Paraguay | Mayo 2025 |
| Indicadores educativos | api.worldbank.org | CC BY 4.0 | Mayo 2025 |

### Calidad de datos detectada
- La columna `anho_cod_geo` del MEC estaba completamente vacía (844 nulos) → eliminada.
- El Banco Mundial tiene valores faltantes en algunos años para Paraguay.
- El MEC solo cubre 2023 → no permite análisis de tendencia interna.

### Privacidad y sesgos
- Datos agregados: no contienen información personal identificable.
- Posible subrepresentación de escuelas rurales que no reportaron al MEC.
- Paraguay y EE.UU. tienen contextos socioeconómicos muy distintos → las diferencias no implican causalidad.

### Medidas de mitigación
- Fuentes oficiales citadas con fecha de descarga.
- Interpretaciones incluyen advertencias sobre limitaciones metodológicas.
- No se publican datos individuales ni sensibles.

## Preguntas de investigación

1. ¿Cómo evolucionó la tasa de matrícula primaria en Paraguay comparado con EE.UU. entre 2000 y 2023?
2. ¿Qué departamentos de Paraguay concentran la mayor cantidad de estudiantes matriculados en 2023?
3. ¿Cuál es la diferencia en gasto en educación (% del PIB) entre Paraguay y EE.UU. a lo largo del tiempo?
4. ¿Qué diferencias existen entre zonas urbanas y rurales en la matrícula escolar paraguaya?
5. ¿Qué problemas de calidad de datos afectan el análisis del dataset MEC?

In [9]:
# GRÁFICO 2 - Matrícula por departamento Paraguay 2023
por_depto = df_mec_clean.groupby('nombre_departamento')['total_matriculados'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colores = ['#0057A8' if i == len(por_depto)-1 else '#4A90D9' for i in range(len(por_depto))]
bars = ax.barh(por_depto.index, por_depto.values, color=colores)

for bar, val in zip(bars, por_depto.values):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_title('Matrícula escolar por departamento — Paraguay 2023',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Total matriculados')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/02_matricula_departamentos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico 2 guardado")

✅ Gráfico 2 guardado


In [10]:
# GRÁFICO 4 - Matrícula urbana vs rural por sector
por_zona_sector = df_mec_clean.groupby(
    ['nombre_zona', 'sector_o_tipo_gestion'])['total_matriculados'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(8, 6))
por_zona_sector.plot(kind='bar', ax=ax,
                     color=['#0057A8', '#CC0000', '#F5A623'],
                     edgecolor='white', width=0.6)

ax.set_title('Matrícula por Zona y Sector de Gestión — Paraguay 2023',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Zona')
ax.set_ylabel('Total matriculados')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(title='Sector')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/04_zona_sector.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico 4 guardado")

✅ Gráfico 4 guardado


In [11]:
# ANÁLISIS DESCRIPTIVO - BANCO MUNDIAL
print("=== ANÁLISIS DESCRIPTIVO - BANCO MUNDIAL ===\n")

# Últimos datos disponibles matrícula primaria
ultimo_py = df_py.dropna(subset=['matricula_primaria']).iloc[-1]
ultimo_us = df_us.dropna(subset=['matricula_primaria']).iloc[-1]
print(f"📊 Matrícula primaria más reciente:")
print(f"   Paraguay ({int(ultimo_py['anio'])}): {ultimo_py['matricula_primaria']:.1f}%")
print(f"   EE.UU.  ({int(ultimo_us['anio'])}): {ultimo_us['matricula_primaria']:.1f}%")
print(f"   Diferencia: {abs(ultimo_py['matricula_primaria'] - ultimo_us['matricula_primaria']):.1f} pp")

# Gasto educación
ult_gasto_py = df_py.dropna(subset=['gasto_educacion_pib']).iloc[-1]
ult_gasto_us = df_us.dropna(subset=['gasto_educacion_pib']).iloc[-1]
print(f"\n📊 Gasto en educación % PIB más reciente:")
print(f"   Paraguay ({int(ult_gasto_py['anio'])}): {ult_gasto_py['gasto_educacion_pib']:.2f}%")
print(f"   EE.UU.  ({int(ult_gasto_us['anio'])}): {ult_gasto_us['gasto_educacion_pib']:.2f}%")

# Resumen MEC
print(f"\n📊 Resumen MEC Paraguay 2023:")
total = df_mec_clean['total_matriculados'].sum()
print(f"   Total matriculados: {total:,}")
por_depto_desc = df_mec_clean.groupby('nombre_departamento')['total_matriculados'].sum().sort_values(ascending=False)
print(f"   Depto. líder: {por_depto_desc.index[0]} ({por_depto_desc.iloc[0]:,})")
print(f"   Depto. menor: {por_depto_desc.index[-1]} ({por_depto_desc.iloc[-1]:,})")
por_zona = df_mec_clean.groupby('nombre_zona')['total_matriculados'].sum()
print(f"   Urbano: {por_zona.get('Urbana', 0):,} | Rural: {por_zona.get('Rural', 0):,}")

=== ANÁLISIS DESCRIPTIVO - BANCO MUNDIAL ===



NameError: name 'df_py' is not defined

In [ ]:
# Limpieza del dataset MEC
print("=== ANÁLISIS DE CALIDAD ===")
print("\nValores nulos por columna:")
print(df_mec.isnull().sum())

print("\nDuplicados:", df_mec.duplicated().sum())

# Eliminar columna anho_cod_geo (está completamente vacía)
df_mec_clean = df_mec.drop(columns=['anho_cod_geo'])

# Agregar columna total matriculados
df_mec_clean['total_matriculados'] = (
    df_mec_clean['cantidad_matriculados_hombre'] + 
    df_mec_clean['cantidad_matriculados_mujer']
)

print("\n✅ Limpieza completada")
print("Columnas finales:", df_mec_clean.columns.tolist())
print("Shape final:", df_mec_clean.shape)
df_mec_clean.head()

=== ANÁLISIS DE CALIDAD ===

Valores nulos por columna:
anio                              0
codigo_departamento               0
nombre_departamento               0
codigo_distrito                   0
nombre_distrito                   0
codigo_zona                       0
nombre_zona                       0
sector_o_tipo_gestion             0
anho_cod_geo                    844
cantidad_matriculados_hombre      0
cantidad_matriculados_mujer       0
dtype: int64

Duplicados: 0

✅ Limpieza completada
Columnas finales: ['anio', 'codigo_departamento', 'nombre_departamento', 'codigo_distrito', 'nombre_distrito', 'codigo_zona', 'nombre_zona', 'sector_o_tipo_gestion', 'cantidad_matriculados_hombre', 'cantidad_matriculados_mujer', 'total_matriculados']
Shape final: (844, 11)


,anio,codigo_departamento,nombre_departamento,codigo_distrito,nombre_distrito,codigo_zona,nombre_zona,sector_o_tipo_gestion,cantidad_matriculados_hombre,cantidad_matriculados_mujer,total_matriculados
0,2023,17,Alto Paraguay,4,Bahía Negra,1,Urbana,Oficial,244,205,449
1,2023,17,Alto Paraguay,4,Bahía Negra,2,Rural,Privado,11,18,29
2,2023,17,Alto Paraguay,4,Bahía Negra,2,Rural,Oficial,88,105,193
3,2023,17,Alto Paraguay,5,Carmelo Peralta,2,Rural,Oficial,347,337,684
4,2023,17,Alto Paraguay,5,Carmelo Peralta,1,Urbana,Oficial,271,223,494


In [12]:
# GRÁFICO 3 - Gasto en educación % PIB Paraguay vs EE.UU.
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df_py['anio'], df_py['gasto_educacion_pib'],
        marker='o', linewidth=2.5, color='#0057A8', label='Paraguay')
ax.plot(df_us['anio'], df_us['gasto_educacion_pib'],
        marker='s', linewidth=2.5, color='#CC0000', label='Estados Unidos')

ax.set_title('Gasto en Educación como % del PIB\nParaguay vs Estados Unidos (2000–2023)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('% del PIB')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/03_gasto_educacion_pib.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico 3 guardado")

NameError: name 'df_py' is not defined

In [ ]:
import urllib.request
import json

print("Descargando datos del Banco Mundial...")

indicadores = {
    'SE.PRM.ENRR': 'matricula_primaria',
    'SE.SEC.ENRR': 'matricula_secundaria',
    'SE.XPD.TOTL.GD.ZS': 'gasto_educacion_pib',
    'SE.PRM.CMPT.ZS': 'finalizacion_primaria'
}

paises = {'PY': 'Paraguay', 'US': 'United States'}
filas = []

for cod_ind, nombre_ind in indicadores.items():
    for cod_pais, nombre_pais in paises.items():
        url = f"https://api.worldbank.org/v2/country/{cod_pais}/indicator/{cod_ind}?date=2000:2023&format=json&per_page=100"
        with urllib.request.urlopen(url) as r:
            data = json.loads(r.read())[1]
        for entry in data:
            if entry['value'] is not None:
                filas.append({
                    'pais': nombre_pais,
                    'anio': int(entry['date']),
                    'indicador': nombre_ind,
                    'valor': entry['value']
                })

df_wb_long = pd.DataFrame(filas)
df_wb = df_wb_long.pivot_table(index=['pais','anio'], columns='indicador', values='valor').reset_index()
df_wb.columns.name = None

df_py = df_wb[df_wb['pais'] == 'Paraguay'].copy().sort_values('anio')
df_us = df_wb[df_wb['pais'] == 'United States'].copy().sort_values('anio')

print("✅ Datos del Banco Mundial descargados")
print("Registros Paraguay:", len(df_py))
print("Registros EE.UU.:", len(df_us))
df_wb.head(10)

Descargando datos del Banco Mundial...


URLError: <urlopen error [Errno 11001] getaddrinfo failed>

In [ ]:
# Guardar datos limpios en data/processed/
df_mec_clean.to_csv('../data/processed/mec_limpio.csv', index=False)
df_wb.to_csv('../data/processed/banco_mundial_limpio.csv', index=False)

print("✅ Archivos guardados en data/processed/")
print("  - mec_limpio.csv")
print("  - banco_mundial_limpio.csv")

✅ Archivos guardados en data/processed/
  - mec_limpio.csv
  - banco_mundial_limpio.csv


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df_py['anio'], df_py['matricula_primaria'],
        marker='o', linewidth=2.5, color='#0057A8', label='Paraguay')
ax.plot(df_us['anio'], df_us['matricula_primaria'],
        marker='s', linewidth=2.5, color='#CC0000', label='Estados Unidos')

ax.set_title('Tasa de Matrícula en Educación Primaria (%)\nParaguay vs Estados Unidos (2000–2023)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Tasa de Matrícula (%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/01_matricula_primaria.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico 1 guardado")

✅ Gráfico 1 guardado


In [ ]:
# Análisis descriptivo del dataset MEC
print("=== ANÁLISIS DESCRIPTIVO - MEC PARAGUAY 2023 ===\n")

# Total matriculados por departamento
por_depto = df_mec_clean.groupby('nombre_departamento')['total_matriculados'].sum().sort_values(ascending=False)
print("Total matriculados por departamento:")
print(por_depto.to_string())

print("\n--- Estadísticas generales ---")
total_py = df_mec_clean['total_matriculados'].sum()
print(f"Total matriculados en Paraguay (2023): {total_py:,}")
print(f"Departamento con más matriculados: {por_depto.index[0]} ({por_depto.iloc[0]:,})")
print(f"Departamento con menos matriculados: {por_depto.index[-1]} ({por_depto.iloc[-1]:,})")

# Por zona
por_zona = df_mec_clean.groupby('nombre_zona')['total_matriculados'].sum()
print(f"\nMatriculados Urbano: {por_zona.get('Urbana', 0):,}")
print(f"Matriculados Rural: {por_zona.get('Rural', 0):,}")

# Por sector
por_sector = df_mec_clean.groupby('sector_o_tipo_gestion')['total_matriculados'].sum()
print(f"\nPor sector:")
print(por_sector.to_string())

=== ANÁLISIS DESCRIPTIVO - MEC PARAGUAY 2023 ===

Total matriculados por departamento:
nombre_departamento
Central             431060
Alto Paraná         205968
Capital             131222
Caaguazú            119543
Itapúa              110996
San Pedro           101868
Cordillera           63802
Concepción           60802
Canindeyú            53829
Paraguarí            50233
Guairá               44570
Caazapá              39060
Amambay              38330
Presidente Hayes     29484
Misiones             28300
Boquerón             17916
Ñeembucú             16516
Alto Paraguay         5054

--- Estadísticas generales ---
Total matriculados en Paraguay (2023): 1,548,553
Departamento con más matriculados: Central (431,060)
Departamento con menos matriculados: Alto Paraguay (5,054)

Matriculados Urbano: 1,053,028
Matriculados Rural: 495,525

Por sector:
sector_o_tipo_gestion
Oficial                  1218256
Privado                   156036
Privado Subvencionado     174261


In [ ]:
# Exportar tablas para usar en Power BI
# Tabla 1: MEC por departamento
resumen_depto = df_mec_clean.groupby('nombre_departamento').agg(
    total_hombres=('cantidad_matriculados_hombre', 'sum'),
    total_mujeres=('cantidad_matriculados_mujer', 'sum'),
    total_matriculados=('total_matriculados', 'sum')
).reset_index().sort_values('total_matriculados', ascending=False)

resumen_depto.to_csv('../data/processed/resumen_departamentos.csv', index=False)

# Tabla 2: Banco Mundial últimos 5 años (2018-2023)
df_wb_5anios = df_wb[df_wb['anio'] >= 2018].copy()
df_wb_5anios.to_csv('../data/processed/banco_mundial_5anios.csv', index=False)

print("✅ Archivos exportados para Power BI:")
print("  - data/processed/resumen_departamentos.csv")
print("  - data/processed/banco_mundial_5anios.csv")
print("\nVista previa resumen departamentos:")
resumen_depto.head(10)

✅ Archivos exportados para Power BI:
  - data/processed/resumen_departamentos.csv
  - data/processed/banco_mundial_5anios.csv

Vista previa resumen departamentos:


,nombre_departamento,total_hombres,total_mujeres,total_matriculados
8,Central,218769,212291,431060
1,Alto Paraná,101608,104360,205968
7,Capital,65427,65795,131222
4,Caaguazú,60246,59297,119543
12,Itapúa,55932,55064,110996
16,San Pedro,51656,50212,101868
10,Cordillera,32575,31227,63802
9,Concepción,30432,30370,60802
6,Canindeyú,26869,26960,53829
14,Paraguarí,25548,24685,50233


In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # evita abrir ventana gráfica
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import wbdata
import datetime
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente
